In [1]:
import tensorflow as tf
import torch
import numpy as np

2026-03-12 18:42:39.980934: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 18:42:40.017776: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-12 18:42:40.787219: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/qduong/Projects/MusicEditLearning/.venv/lib/python3.12/site-packages/keras/src/export

In [2]:
# Device initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data paths
test_path = "data/test/*.tfrecord"
train_path = "data/train/*.tfrecord"
val_path = "data/validation/*.tfrecord"

In [3]:
def parse_seq_example(example_proto):
    """Parse TFRecord example into pitch sequence."""
    context_features = {}
    sequence_features = {
        "pitch_seq": tf.io.VarLenFeature(dtype=tf.int64),
    }
    _, sequence = tf.io.parse_single_sequence_example(
        example_proto,
        context_features=context_features,
        sequence_features=sequence_features
    )
    pitch_seq = tf.sparse.to_dense(sequence["pitch_seq"])
    pitch_seq = tf.reshape(pitch_seq, [-1])
    return pitch_seq

In [20]:
raw_data = tf.data.TFRecordDataset(tf.io.gfile.glob(train_path))
dataset = raw_data.map(parse_seq_example)

In [21]:
for parsed_record in dataset.take(1): # Accessing one element
    print(repr(parsed_record))


<tf.Tensor: shape=(64,), dtype=int64, numpy=
array([129,  59,  59, 129, 129, 129, 129, 129, 129, 129, 129, 129, 129,
        63, 128, 129, 129, 129, 129,  59,  59, 129, 129,  58,  59, 129,
        61,  63,  64, 129, 129,  68, 128, 129, 129,  66,  66, 129, 129,
       129, 129, 129, 129, 129, 129, 129, 129, 129,  75, 129, 129,  70,
        70,  70,  70, 129, 129,  70, 129, 129,  72,  74, 129,  75])>


In [22]:
counter = 0
for i in dataset.as_numpy_iterator():
    counter += 1

print(f"Total records in dataset: {counter}")

Total records in dataset: 10126676


2026-03-12 14:33:08.931199: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
# testing set: 22060 
# validation set: 70908
# training set: 10126676

# Conversion parameters
# num_samples = 200000  # Adjust this number as needed (set to None for all samples)
# output_file = 'data/train_data.npy'  # Output file path

# Choose dataset: 'train', 'test', or 'validation'
# dataset_choice = 'train'  # Change this to select different datasets

In [7]:
def sample_dataset(dataset_choice, num_samples=1000, output_file=None):

    if dataset_choice == 'train':
        dataset_path = train_path
    elif dataset_choice == 'test':
        dataset_path = test_path
    elif dataset_choice == 'val':
        dataset_path = val_path
    else:
        raise ValueError("Invalid dataset_choice. Choose 'train', 'test', or 'validation'")

    # Create dataset
    full_dataset = tf.data.TFRecordDataset(tf.io.gfile.glob(dataset_path)).map(parse_seq_example)

    # Collect samples
    samples = []
    take_dataset = full_dataset.take(num_samples) if num_samples is not None else full_dataset

    for i, parsed_record in enumerate(take_dataset):
        samples.append(parsed_record.numpy())
        if (i + 1) % 1000 == 0:
            print(f"Collected {i + 1} samples")

    print(f"Total collected: {len(samples)} samples")

    # Convert to numpy array and save
    arr = np.array(samples)
    print(f"Array shape: {arr.shape}")
    print(f"Data type: {arr.dtype}")

    # Verify all sequences have the expected length
    sequence_lengths = [len(seq) for seq in arr]
    unique_lengths = set(sequence_lengths)
    if len(unique_lengths) == 1:
        print(f"All sequences have length {list(unique_lengths)[0]}")
    else:
        print(f"Warning: Found sequences with lengths {sorted(unique_lengths)}")

    np.save(output_file, arr)
    print(f"Saved to {output_file}")

def verify_saved_data(file_path):
    loaded_data = np.load(file_path, allow_pickle=True)
    print(f"Loaded data shape: {loaded_data.shape}")
    print(f"Data type: {loaded_data.dtype}")

    # Verify the first few entries
    for i in range(min(5, len(loaded_data))):
        print(f"Sample {i}: {loaded_data[i]}")

    tensor = torch.from_numpy(loaded_data)
    print(f"PyTorch tensor shape: {tensor.shape}")
    print(f"PyTorch tensor dtype: {tensor.dtype}")

In [8]:
sample_dataset('train', num_samples=200000, output_file='data/train_data.npy')

Collected 1000 samples
Collected 2000 samples
Collected 3000 samples
Collected 4000 samples
Collected 5000 samples
Collected 6000 samples
Collected 7000 samples
Collected 8000 samples
Collected 9000 samples
Collected 10000 samples
Collected 11000 samples
Collected 12000 samples
Collected 13000 samples
Collected 14000 samples
Collected 15000 samples
Collected 16000 samples
Collected 17000 samples
Collected 18000 samples
Collected 19000 samples
Collected 20000 samples
Collected 21000 samples
Collected 22000 samples
Collected 23000 samples
Collected 24000 samples
Collected 25000 samples
Collected 26000 samples
Collected 27000 samples
Collected 28000 samples
Collected 29000 samples
Collected 30000 samples
Collected 31000 samples
Collected 32000 samples
Collected 33000 samples
Collected 34000 samples
Collected 35000 samples
Collected 36000 samples
Collected 37000 samples
Collected 38000 samples
Collected 39000 samples
Collected 40000 samples
Collected 41000 samples
Collected 42000 samples
C

In [11]:
verify_saved_data('data/test_data.npy')

Loaded data shape: (20000, 64)
Data type: int64
Sample 0: [129 129 129 129 129 129 129 129  62 128 128 128 129 129  62 128 128 128
 128 128 128 129 129 129  62 128 128 128 129 129  62 128 128 128 128 128
 128 128 129 129  61 128 128 128 129 129  61 128 128 128 128 128 128 128
 129 129  61 128 128 128 129 129  61 128]
Sample 1: [ 73  73  73  73 129  73 129  71 129 129 129 129  71  68  71 129  71  71
  71  71 129 129 129 129 129 129 129 129  71  71 129  71  73  73  73  73
 129  73 129  71 129 129 129 129  71  68  71 129  72  72  72  72 129 129
 129 129 129 129 129 129  72  72 129  72]
Sample 2: [ 65 128 128 128 128 128 128 128 128 128 128 128 128 129 129 129  69 128
 128 128 128 128  67 128  65 128 128 128  69 128 128 128  65 128 128 128
 128 128 128 128 128 128 128 128 128 129 129 129  69 128 128 128 128 128
  67 128  65 128 128 128  69 128 128 128]
Sample 3: [ 33 128 128 128 128  33 128 128 128 128  26 128 128 128 128 128  26 128
  38 128 128 128 128 128  38 128 128 128 128 128 128 128